In [ ]:
# Install dependencies
# !pip install ultralytics opencv-python pillow matplotlib

## Bounding Box Fundamentals

In [ ]:
import numpy as np
from typing import List, Tuple
from dataclasses import dataclass

@dataclass
class BoundingBox:
    """Bounding box representation"""
    x: float  # Top-left x
    y: float  # Top-left y
    width: float
    height: float
    confidence: float
    class_id: int
    class_name: str
    
    @property
    def x_min(self) -> float:
        return self.x
    
    @property
    def y_min(self) -> float:
        return self.y
    
    @property
    def x_max(self) -> float:
        return self.x + self.width
    
    @property
    def y_max(self) -> float:
        return self.y + self.height
    
    @property
    def area(self) -> float:
        return self.width * self.height
    
    def to_xyxy(self) -> Tuple[float, float, float, float]:
        """Convert to [x_min, y_min, x_max, y_max] format"""
        return (self.x_min, self.y_min, self.x_max, self.y_max)
    
    def to_xywh(self) -> Tuple[float, float, float, float]:
        """Convert to [x, y, width, height] format"""
        return (self.x, self.y, self.width, self.height)

def compute_iou(box1: BoundingBox, box2: BoundingBox) -> float:
    """Compute Intersection over Union (IoU)"""
    # Intersection coordinates
    x_min = max(box1.x_min, box2.x_min)
    y_min = max(box1.y_min, box2.y_min)
    x_max = min(box1.x_max, box2.x_max)
    y_max = min(box1.y_max, box2.y_max)
    
    # Intersection area
    if x_max < x_min or y_max < y_min:
        return 0.0
    
    intersection = (x_max - x_min) * (y_max - y_min)
    
    # Union area
    union = box1.area + box2.area - intersection
    
    return intersection / union if union > 0 else 0.0

# Test IoU
box1 = BoundingBox(10, 10, 50, 50, 0.9, 0, "person")
box2 = BoundingBox(30, 30, 50, 50, 0.8, 0, "person")
box3 = BoundingBox(100, 100, 50, 50, 0.7, 1, "car")

print(f"IoU(box1, box2) = {compute_iou(box1, box2):.3f}  # Overlapping")
print(f"IoU(box1, box3) = {compute_iou(box1, box3):.3f}  # Non-overlapping")

## Non-Maximum Suppression (NMS)

In [ ]:
def non_max_suppression(boxes: List[BoundingBox], iou_threshold: float = 0.5) -> List[BoundingBox]:
    """Apply Non-Maximum Suppression to remove duplicate detections"""
    if not boxes:
        return []
    
    # Sort by confidence (highest first)
    boxes = sorted(boxes, key=lambda b: b.confidence, reverse=True)
    
    keep = []
    
    while boxes:
        # Take box with highest confidence
        best_box = boxes.pop(0)
        keep.append(best_box)
        
        # Remove boxes with high IoU
        boxes = [
            box for box in boxes
            if compute_iou(best_box, box) < iou_threshold
            or box.class_id != best_box.class_id  # Different class
        ]
    
    return keep

# Test NMS
detections = [
    BoundingBox(10, 10, 50, 50, 0.95, 0, "person"),
    BoundingBox(12, 12, 50, 50, 0.90, 0, "person"),  # Similar to first
    BoundingBox(15, 15, 50, 50, 0.85, 0, "person"),  # Similar to first
    BoundingBox(100, 100, 50, 50, 0.92, 1, "car"),
]

filtered = non_max_suppression(detections, iou_threshold=0.5)

print(f"Before NMS: {len(detections)} boxes")
print(f"After NMS:  {len(filtered)} boxes")
print("\nKept boxes:")
for box in filtered:
    print(f"  {box.class_name} @ ({box.x:.0f}, {box.y:.0f}): {box.confidence:.2f}")

## YOLO Object Detector

In [ ]:
# YOLO with Ultralytics (requires installation)
'''
from ultralytics import YOLO
import cv2

# Load YOLOv8 model
model = YOLO('yolov8n.pt')  # nano model (fastest)
# Other options: yolov8s, yolov8m, yolov8l, yolov8x

# Detect objects in image
results = model('path/to/image.jpg')

# Process results
for result in results:
    boxes = result.boxes  # Bounding boxes
    for box in boxes:
        x1, y1, x2, y2 = box.xyxy[0]
        conf = box.conf[0]
        cls = box.cls[0]
        print(f"Detected {model.names[int(cls)]} at ({x1:.0f}, {y1:.0f}) with confidence {conf:.2f}")

# Real-time detection from webcam
cap = cv2.VideoCapture(0)
while True:
    ret, frame = cap.read()
    if not ret:
        break
    
    results = model(frame, stream=True)
    
    for result in results:
        annotated = result.plot()  # Draw boxes
        cv2.imshow('YOLO', annotated)
    
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()
'''

print("YOLOv8 detection example (commented - requires ultralytics)")
print("\nYOLO Models:")
print("  yolov8n - Nano (fastest, least accurate)")
print("  yolov8s - Small")
print("  yolov8m - Medium")
print("  yolov8l - Large")
print("  yolov8x - Extra Large (slowest, most accurate)")

## Custom Object Detector

In [ ]:
class ObjectDetector:
    """Simple object detection wrapper"""
    
    def __init__(self, conf_threshold: float = 0.5, iou_threshold: float = 0.5):
        self.conf_threshold = conf_threshold
        self.iou_threshold = iou_threshold
        self.class_names = self._load_class_names()
    
    def _load_class_names(self) -> List[str]:
        """Load COCO class names"""
        # COCO 80 classes (subset shown)
        return [
            'person', 'bicycle', 'car', 'motorcycle', 'airplane',
            'bus', 'train', 'truck', 'boat', 'traffic light',
            'cat', 'dog', 'horse', 'sheep', 'cow'
            # ... 65 more classes
        ]
    
    def detect(self, image: np.ndarray) -> List[BoundingBox]:
        """Detect objects in image"""
        # Simulate detections
        raw_detections = self._simulate_detections()
        
        # Filter by confidence
        filtered = [d for d in raw_detections if d.confidence >= self.conf_threshold]
        
        # Apply NMS
        final_detections = non_max_suppression(filtered, self.iou_threshold)
        
        return final_detections
    
    def _simulate_detections(self) -> List[BoundingBox]:
        """Simulate raw model output"""
        # In production: actual model inference
        return [
            BoundingBox(50, 50, 100, 150, 0.95, 0, "person"),
            BoundingBox(52, 52, 100, 150, 0.92, 0, "person"),  # Duplicate
            BoundingBox(200, 100, 80, 60, 0.88, 2, "car"),
            BoundingBox(150, 300, 50, 50, 0.76, 10, "cat"),
            BoundingBox(400, 200, 120, 100, 0.42, 3, "motorcycle"),  # Low conf
        ]
    
    def visualize(self, image: np.ndarray, boxes: List[BoundingBox]) -> np.ndarray:
        """Draw boxes on image"""
        # In production: use cv2.rectangle() to draw boxes
        print(f"\nWould draw {len(boxes)} boxes on image:")
        for box in boxes:
            print(f"  {box.class_name}: ({box.x:.0f}, {box.y:.0f}, {box.width:.0f}, {box.height:.0f}) - {box.confidence:.2f}")
        return image

# Test detector
detector = ObjectDetector(conf_threshold=0.5, iou_threshold=0.5)

# Dummy image
image = np.zeros((640, 640, 3), dtype=np.uint8)
detections = detector.detect(image)

print(f"\nDetected {len(detections)} objects:")
for det in detections:
    print(f"  {det.class_name}: {det.confidence:.2%} at ({det.x:.0f}, {det.y:.0f})")

# Visualize
annotated = detector.visualize(image, detections)

## Evaluation Metrics

In [ ]:
def compute_precision_recall(predictions: List[BoundingBox], 
                              ground_truth: List[BoundingBox],
                              iou_threshold: float = 0.5) -> Tuple[float, float]:
    """Compute precision and recall"""
    true_positives = 0
    matched_gt = set()
    
    for pred in predictions:
        best_iou = 0
        best_gt_idx = -1
        
        for idx, gt in enumerate(ground_truth):
            if gt.class_id != pred.class_id:
                continue
            
            iou = compute_iou(pred, gt)
            if iou > best_iou:
                best_iou = iou
                best_gt_idx = idx
        
        if best_iou >= iou_threshold and best_gt_idx not in matched_gt:
            true_positives += 1
            matched_gt.add(best_gt_idx)
    
    false_positives = len(predictions) - true_positives
    false_negatives = len(ground_truth) - len(matched_gt)
    
    precision = true_positives / (true_positives + false_positives) if predictions else 0
    recall = true_positives / (true_positives + false_negatives) if ground_truth else 0
    
    return precision, recall

def compute_ap(precisions: List[float], recalls: List[float]) -> float:
    """Compute Average Precision (AP)"""
    # Sort by recall
    sorted_indices = np.argsort(recalls)
    recalls = np.array(recalls)[sorted_indices]
    precisions = np.array(precisions)[sorted_indices]
    
    # Compute AP using 11-point interpolation
    ap = 0
    for t in np.arange(0, 1.1, 0.1):
        if np.sum(recalls >= t) == 0:
            p = 0
        else:
            p = np.max(precisions[recalls >= t])
        ap += p / 11
    
    return ap

# Test metrics
pred_boxes = [
    BoundingBox(10, 10, 50, 50, 0.9, 0, "person"),
    BoundingBox(100, 100, 50, 50, 0.8, 1, "car"),
]

gt_boxes = [
    BoundingBox(12, 12, 50, 50, 1.0, 0, "person"),
    BoundingBox(102, 102, 50, 50, 1.0, 1, "car"),
    BoundingBox(200, 200, 50, 50, 1.0, 2, "dog"),  # Missed
]

precision, recall = compute_precision_recall(pred_boxes, gt_boxes)
print(f"Precision: {precision:.2%}")
print(f"Recall: {recall:.2%}")
print(f"F1 Score: {2 * precision * recall / (precision + recall):.2%}")

## Production Deployment

In [ ]:
import time
from collections import deque

class ProductionDetector:
    """Production-ready object detector"""
    
    def __init__(self, model_name: str = "yolov8n"):
        self.model_name = model_name
        self.detector = ObjectDetector()
        self.stats = {
            "total_images": 0,
            "total_detections": 0,
            "avg_inference_time": 0,
            "fps_history": deque(maxlen=30)
        }
    
    def detect_with_timing(self, image: np.ndarray) -> Tuple[List[BoundingBox], float]:
        """Detect with performance tracking"""
        start = time.time()
        detections = self.detector.detect(image)
        inference_time = time.time() - start
        
        # Update stats
        self.stats["total_images"] += 1
        self.stats["total_detections"] += len(detections)
        self.stats["fps_history"].append(1 / inference_time if inference_time > 0 else 0)
        self.stats["avg_inference_time"] = (
            (self.stats["avg_inference_time"] * (self.stats["total_images"] - 1) + inference_time)
            / self.stats["total_images"]
        )
        
        return detections, inference_time
    
    def get_performance_stats(self) -> dict:
        """Get performance statistics"""
        avg_fps = np.mean(self.stats["fps_history"]) if self.stats["fps_history"] else 0
        
        return {
            "total_images": self.stats["total_images"],
            "total_detections": self.stats["total_detections"],
            "avg_detections_per_image": (
                self.stats["total_detections"] / max(self.stats["total_images"], 1)
            ),
            "avg_inference_time_ms": self.stats["avg_inference_time"] * 1000,
            "avg_fps": avg_fps
        }

# Test production detector
prod_detector = ProductionDetector()

# Process images
for i in range(10):
    image = np.zeros((640, 640, 3), dtype=np.uint8)
    detections, time_ms = prod_detector.detect_with_timing(image)

# Print stats
stats = prod_detector.get_performance_stats()
print("\nPerformance Statistics:")
print(f"  Total Images: {stats['total_images']}")
print(f"  Total Detections: {stats['total_detections']}")
print(f"  Avg Detections/Image: {stats['avg_detections_per_image']:.1f}")
print(f"  Avg Inference Time: {stats['avg_inference_time_ms']:.2f}ms")
print(f"  Avg FPS: {stats['avg_fps']:.1f}")

## Best Practices

### 1. Model Selection
- **Real-time (>30 FPS)**: YOLOv8n, YOLOv8s
- **Balanced**: YOLOv8m, DETR
- **High accuracy**: YOLOv8x, Faster R-CNN
- **Edge devices**: YOLOv8n with TensorRT

### 2. Hyperparameters
- **Confidence threshold**: 0.25-0.5 (lower = more detections)
- **IoU threshold (NMS)**: 0.45-0.65 (lower = fewer duplicates)
- **Image size**: 640×640 (YOLO), can go lower for speed

### 3. Training Tips
- Use data augmentation (mosaic, mixup)
- Balance classes with weighted sampling
- Train on high-resolution images
- Freeze backbone initially, then fine-tune

### 4. Optimization
- Convert to ONNX/TensorRT for faster inference
- Batch processing when possible
- Resize images to smaller sizes (320×320, 416×416)
- Use FP16 precision on GPUs

## Common Use Cases

- **Autonomous vehicles**: Detect pedestrians, cars, traffic signs
- **Surveillance**: People counting, intrusion detection
- **Retail**: Product detection, shelf monitoring
- **Manufacturing**: Defect detection, quality control
- **Agriculture**: Crop monitoring, pest detection

## Key Takeaways

✅ Object detection = Classification + Localization

✅ YOLO is the best choice for real-time applications

✅ NMS removes duplicate detections

✅ IoU measures box overlap (used in NMS and evaluation)

✅ mAP (mean Average Precision) is the standard metric

✅ Balance speed vs accuracy based on use case

---

**Next: [03_clip_embeddings.ipynb](03_clip_embeddings.ipynb) - Multimodal understanding**